### Author: Tanwir Silar

# Case Study - Freddie Mac Bonds


## 1. Pricing the Callable Bond


### Data

Use the data from the following files.
* `../data/callable_bonds_2025-02-13.xlsx`
* `../data/discount_curve_2025-02-13.xlsx`


The data contains info on the following bonds.

`Callable`
* `FHLMC 4.41 01/28/30` is a callable bond, and it is the primary object of our analysis.


In [1]:
FILE_BOND = '../data/callable_bonds_2025-02-13.xlsx'
FILE_CURVE = '../data/discount_curve_2025-02-13.xlsx'

KEY_CALLABLE = 'FHLMC 4.41 01/28/30'

### Bond Info


In [2]:
import pandas as pd
import numpy as np
import math
from scipy.stats import norm

info = pd.read_excel(FILE_BOND,sheet_name='info').set_index('info')
info_core = info[[KEY_CALLABLE]]
info_core.style.format('{:.2%}',subset=pd.IndexSlice[["Cpn Rate"], :]).format('{:,.0f}',subset=pd.IndexSlice[["Amount Issued"], :]).format('{:%Y-%m-%d}',subset=pd.IndexSlice[["Date Quoted","Date Issued","Date Matures","Date Next Call","Date of First Possible Call"], :])

,FHLMC 4.41 01/28/30
info,
CUSIP,3134HA4V2
Issuer,FREDDIE MAC
Maturity Type,CALLABLE
Issuer Industry,GOVT AGENCY
Amount Issued,"10,000,000"
Cpn Rate,4.41%
Cpn Freq,2
Date Quoted,2025-02-13
Date Issued,2025-01-28


### Quoted Values


In [3]:
quotes = pd.read_excel(FILE_BOND,sheet_name='quotes').set_index('quotes')
quotes_core = quotes[[KEY_CALLABLE]]
quotes_core.style.format('{:.2f}', subset=pd.IndexSlice[quotes.index[1:], :]).format('{:%Y-%m-%d}', subset=pd.IndexSlice['Date Quoted', :])

,FHLMC 4.41 01/28/30
quotes,
Date Quoted,2025-02-13
TTM,4.96
Clean Price,99.89
Dirty Price,100.09
Accrued Interest,0.20
YTM Call,4.45
YTM Maturity,4.43
Duration,4.50
Modified Duration,4.40


### Discount Curves


In [4]:
discs = pd.read_excel(FILE_CURVE,sheet_name='discount curve').set_index('ttm')
display(discs.head())
display(discs.tail())

,maturity date,spot rate,discount
ttm,,,
0.5,2025-08-13,0.043743,0.978597
1.0,2026-02-13,0.042890,0.958451
1.5,2026-08-13,0.042238,0.939228
2.0,2027-02-13,0.041843,0.920515
2.5,2027-08-13,0.041632,0.902117


,maturity date,spot rate,discount
ttm,,,
28.0,2053-02-13,0.040185,0.328231
28.5,2053-08-13,0.040051,0.322978
29.0,2054-02-13,0.039916,0.317851
29.5,2054-08-13,0.039791,0.312766
30.0,2055-02-13,0.039665,0.307802


### 1.1.

Use the discount curve data to price both the `callable` and `reference` bonds.

Also calculate the price of the `hypothetical` bonds, where we consider a non-callable version of the callable bond with 
* maturity unchanged
* maturity at the call date.


### Forward Bond Price

For Black's formula, we need the **forward** bond price. 

This is straightforward to calculate, though it requires a few assumptions. 

$$P_{\text{forward}}(T_\text{option}\to T) = \frac{P(T) - \sum_{i=1}^n Z(T_i)C_i}{Z(T_\text{option})}$$

where $n$ denotes the number of cashflows (coupons) between $t$ and $T_{\text{option}}$, and $C_i$ denote the size of those coupons.

In [98]:
# 1.1

def interpolate_discount_factor_rates(discs, t, key = 'discount'):
    # print(f"Interpolating discount factor for t={t}")
    if t in discs.index:
        return discs.loc[t, key]
    else:
        # find the two nearest points for interpolation
        lower_index = discs.index[discs.index < t].max()
        upper_index = discs.index[discs.index > t].min()
        
        # linear interpolation
        if np.isnan(lower_index) or np.isnan(upper_index) and not (np.isnan(lower_index) and np.isnan(upper_index)):
            print(f"Using nearest discount factor for t={t}")
            if np.isnan(lower_index):
                return discs.loc[upper_index, key]
            else:
                return discs.loc[lower_index, key]
        elif np.isnan(lower_index) and np.isnan(upper_index):
            raise ValueError(f"No discount factors available for time t={t}")
        
        lower = discs.loc[lower_index, key]
        upper = discs.loc[upper_index, key]
        
        interpolated_val = lower + (upper - lower) * (t - lower_index) / (upper_index - lower_index)
        return interpolated_val
    
def forward_bond_price(cpn_rate, cpn_freq, T, N, iv, P_Tb, discs):
    # todo: if given the current date, calculate the coupon dates and use the discount factors for those dates instead of assuming they are equally spaced
    # calculate the discount factors by interpolating the discount rates given 

    '''
        T = tte of option in years
        N = par value of bond
        discs = discount curve for 0, T/n, 2T/n, ..., T
        P_Tb = bond price at time of expiry of bond 

    '''
    # print(f"Calculating forward bond price with T_option={T}, N={N}, cpn_rate={cpn_rate}, cpn_freq={cpn_freq}, iv={iv}, P_Tb={P_Tb}")

    forward_price = P_Tb
    for i in range(1, int(T*cpn_freq)+1):
        forward_price -= cpn_rate * N / cpn_freq * interpolate_discount_factor_rates(discs, i/cpn_freq)    
    forward_price = forward_price/interpolate_discount_factor_rates(discs, T)
    return forward_price

def calc_option_price(forward_price, K, T, iv_rates, discount_factor, info, quotes, option_type='call'):
    # print(f"Calculating option price with forward_price={forward_price}, K={K}, T={T}, iv={iv_rates}, discount_factor={discount_factor}, option_type={option_type}")

    date_next_call = info.loc['Date Next Call', KEY_CALLABLE]
    date_quoted = info.loc['Date Quoted', KEY_CALLABLE]

    time_to_maturity_option = (date_next_call - date_quoted).days / 365
    duration = quotes.loc['Duration', KEY_CALLABLE]
    instantaneous_forward_rate_approx = interpolate_discount_factor_rates(discs, time_to_maturity_option, key = 'spot rate')
    iv_fwd_price = duration * iv_rates * instantaneous_forward_rate_approx


    d1 = (np.log(forward_price / K) + (iv_fwd_price**2 / 2) * T) / (iv_fwd_price * math.sqrt(T))
    d2 = d1 - iv_fwd_price * math.sqrt(T)
    price = 0
    if option_type == 'call':
        price = forward_price * norm.cdf(d1) - K * norm.cdf(d2)
    elif option_type == 'put':
        price = K * norm.cdf(-d2) - forward_price * norm.cdf(-d1)
    price *= discount_factor
    return price

def price_vanilla_bond(cpn_rate, cpn_freq, T, N, discs):
    # print(f"Calculating vanilla bond price with cpn_rate={cpn_rate}, cpn_freq={cpn_freq}, T={T}, N={N}")
    price = 0
    for i in range(1, int(T*cpn_freq)+1):
        price += cpn_rate * N / cpn_freq * interpolate_discount_factor_rates(discs, i/cpn_freq)
    price += N * interpolate_discount_factor_rates(discs, T)
    return price

def price_callable_bond(info, quotes, discs, key):
    cpn_rate = info.loc['Cpn Rate', key]
    cpn_freq = info.loc['Cpn Freq', key]
    date_quoted = info.loc['Date Quoted', key]
    date_matures = info.loc['Date Matures', key]
    date_next_call = info.loc['Date Next Call', key]
    N = 100
    iv = quotes.loc['Implied Vol', key] / 100

    vanilla_bond_price = price_vanilla_bond(cpn_rate, cpn_freq, (date_matures - date_quoted).days / 365, N, discs)

    forward_price = forward_bond_price(cpn_rate, cpn_freq, (date_next_call - date_quoted).days / 365, N, iv, vanilla_bond_price, discs)
    option_price = calc_option_price(forward_price, N, (date_next_call - date_quoted).days / 365, iv, interpolate_discount_factor_rates(discs, (date_next_call - date_quoted).days / 365), info, quotes, option_type='call')
    callable_price = vanilla_bond_price - option_price

    return {
        'callable_price': callable_price, 
        'option_price': option_price,
        'vanilla_bond_price': vanilla_bond_price,
        'forward_price': forward_price
    }

In [80]:
results = price_callable_bond(info, quotes, discs, KEY_CALLABLE)
callable_price = results['callable_price']
vanilla_bond_price = results['vanilla_bond_price']
option_price = results['option_price']
forward_price = results['forward_price']

print(f"Callable Bond Price: {results['callable_price']:.2f}")
print(f"Vanilla Bond Price: {results['vanilla_bond_price']:.2f}")
print(f"Option Price: {results['option_price']:.2f}")
print(f"Forward Price: {results['forward_price']:.2f}")

Callable Bond Price: 96.52
Vanilla Bond Price: 99.57
Option Price: 3.05
Forward Price: 100.72


### 1.2.

Calculate the forward price of the `hypothetical` bond as of the date that the `callable` bond can be exercised.

Use the information from the discount curve (and associated forward curve) to calculate this forward price.


In [81]:
print(f"forward price: {forward_price:.2f}")

forward price: 100.72


### 1.3.

The provided implied vol corresponds to the implied vol of the **rate**. Specifically,
* the forward rate corresponding to the time of expiration.
* continuously compounded.

Use the duration approximation to get the approximate implied vol corresponding to the forward price.

$$\sigma_{\text{bond fwd price}} \approx D \times \sigma_{\text{fwd rate}}\times f(T_1)$$

where $f(T_1)$ is the continuously-compounded (instantaneous) forward rate at time $T_1$.
* If you're struggling with the forward rate calc, just usse the provided spot rate at $T_1$; it will be a close approximation in this example.
* In this approximation, use the quoted duration from the table. (Yes, this is a bit circular, but we don't want to get bogged down with a duration calculation at this point.)

Report the implied vol of the bond's forward price.


In [82]:
def calc_implied_vol_forward_price(info, quotes, discs, key):
    date_next_call = info.loc['Date Next Call', key]
    date_quoted = info.loc['Date Quoted', key]
    iv_rates = quotes.loc['Implied Vol', key] / 100

    time_to_maturity_option = (date_next_call - date_quoted).days / 365
    duration = quotes.loc['Duration', key]
    instantaneous_forward_rate_approx = interpolate_discount_factor_rates(discs, time_to_maturity_option, key = 'spot rate')
    iv_fwd_price = duration * iv_rates * instantaneous_forward_rate_approx
    return iv_fwd_price
iv_fwd_price = calc_implied_vol_forward_price(info, quotes, discs, KEY_CALLABLE)

print(f'implied vol of bond\'s forward price: {iv_fwd_price}')

implied vol of bond's forward price: 0.0445680635361678


### 1.4.

For the `callable` bond, report Black's value of the embedded call option.
* Use this to report the value of the `callable` bond.
* How does it compare to the actual market price?

For the calculation of the option, use...
* the quoted `Implied Vol` calculated above.
* forward price of the `hypothetical` bond calculated above.
* provided discount factor

#### Simplifications
Note that in this calculation we are making a few simplifications.
* We are simplifying that the `callable` bond is European exercise with an exercise date as reported in `Date Next Call` above. 
* In reality, it is Bermudan, with quarterly exercise dates after the first exercise date.
* The time-to-exercise is not a round number, but you only have discount factors at rounded time-to-maturities. Just use the closest discount factor.


In [83]:
print(f'option price of the callable bond: {results["option_price"]:.2f}')
print(f'value of callable bond: {callable_price}')
market_price = quotes.loc['Clean Price', KEY_CALLABLE]
print(f'actual market price: {market_price:.2f} vs calculated price: {callable_price:.2f}')

option price of the callable bond: 3.05
value of callable bond: 96.51922936055577
actual market price: 99.89 vs calculated price: 96.52


### 1.5.

Calculate the YTM of the callable bond, assuming that...
* it can never be called. (This is the `hypothetical` bond we analyzed above.)
* it will certainly be called.

How do these compare to the quoted YTM Called and YTM Maturity in the table?


In [84]:
cpn_rate = info.loc['Cpn Rate', KEY_CALLABLE]
cpn_freq = info.loc['Cpn Freq', KEY_CALLABLE]
date_matures = info.loc['Date Matures', KEY_CALLABLE]
date_quoted = info.loc['Date Quoted', KEY_CALLABLE]
N = 100 # face value of bond

In [85]:
from scipy.optimize import root_scalar
def calc_YTM(price, cpn_rate, cpn_freq, T, N):
    # calculate the yield to maturity of a bond given its price and other characteristics
    # use a numerical method to solve for the yield that equates the present value of the bond's cash flows to its price
    def pv_func(y):
        pv = 0
        for i in range(1, int(T*cpn_freq)+1):
            pv += cpn_rate * N / cpn_freq / (1 + y)**(i)
        pv += N / (1 + y)**(T*cpn_freq)
        return pv - price
    result = root_scalar(pv_func, bracket=[0.001, 1.0])
    ytm = result.root * cpn_freq # annualize the yield
    return ytm




In [86]:

ytm_never_called = calc_YTM(callable_price, cpn_rate, cpn_freq, (date_matures - date_quoted).days / 365, N)
ytm_certainly_called = calc_YTM(callable_price, cpn_rate, cpn_freq, (date_next_call - date_quoted).days / 365, N)

print(f"Yield to maturity if never called: {ytm_never_called:.4f}")
print(f"Yield to maturity if certainly called at first call date: {ytm_certainly_called:.4f}")

Yield to maturity if never called: 0.0484
Yield to maturity if certainly called at first call date: 0.0505


### 1.6.

Calculate the duration of...
* the `hypothetical` bond
* the `callable` bond

How do these compare to the quoted duration in the table?

For the callable bond, calculate duration numerically by modifying the spot rates up and down by 1bp and seeing how it changes the valuation of parts `1.1`-`1.3`.


In [87]:
def calc_next_prev_coupon_date(info, key):
    date_quoted = info.loc['Date Quoted', key]
    date_issued = info.loc['Date Issued', key]
    date_matures = info.loc['Date Matures', key]
    cpn_freq = info.loc['Cpn Freq', key]

    factor = int((date_quoted-date_issued).days / (365/cpn_freq)) + 1
    next_coupon_date = date_issued + pd.DateOffset(months = 12/cpn_freq* factor)
    prev_coupon_date = date_issued + pd.DateOffset(months = 12/cpn_freq* (factor-1))

    if prev_coupon_date < date_issued:
        prev_coupon_date = None
    if next_coupon_date > date_matures:
        next_coupon_date = None
        
    return prev_coupon_date, next_coupon_date

info2 = pd.DataFrame({
    'Date Quoted': [pd.to_datetime('2025-02-13')],
    'Date Issued': [pd.to_datetime('2025-01-31')],
    'Date Matures': [pd.to_datetime('2030-01-28')],
    'Cpn Freq': [2]
}, index=[KEY_CALLABLE]).T
pcd, ncd = calc_next_prev_coupon_date(info2, KEY_CALLABLE)
print(pcd, ncd)

2025-01-31 00:00:00 2025-07-31 00:00:00


In [88]:
def calc_modified_duration(price, cpn_rate, cpn_freq, T, N, ytm):
    # calculate the modified duration of a bond given its price and other characteristics
    # use the formula for modified duration which is the Macaulay duration divided by (1 + ytm/cpn_freq)
    def calc_macaulay_duration(cpn_rate, cpn_freq, T, N, ytm):
        macaulay_duration = 0
        for i in range(1, int(T*cpn_freq)+1):
            macaulay_duration += i * cpn_rate * N / cpn_freq / (1 + ytm/cpn_freq)**(i)
        macaulay_duration += T * N / (1 + ytm/cpn_freq)**(T*cpn_freq)
        macaulay_duration /= price
        return macaulay_duration
    macaulay_duration = calc_macaulay_duration(cpn_rate, cpn_freq, T, N, ytm)
    modified_duration = macaulay_duration/(1 + ytm/cpn_freq)
    return modified_duration



def calc_duration_from_shocks(price, info, quotes, discs, shock = 0.0001):
    # calculate the duration of a bond given its price and two shocked prices
    # use the formula for duration which is (price_down - price_up) / (2 * price * shock)
    discs_copy = discs.copy()
    discs_copy['spot rate up'] = discs_copy['spot rate'] + shock
    discs_copy['discount up'] = np.exp(-discs_copy['spot rate up'] * discs_copy.index)
    discs_copy['spot rate down'] = discs_copy['spot rate'] - shock
    discs_copy['discount down'] = np.exp(-discs_copy['spot rate down'] * discs_copy.index)

    # print(discs_copy.index)
    discs_up = discs_copy[['spot rate up']].rename(columns={'spot rate up': 'spot rate'})
    discs_up['discount'] = np.exp(-discs_up['spot rate'] * discs_up.index)
    discs_down = discs_copy[['spot rate down']].rename(columns={'spot rate down': 'spot rate'})
    discs_down['discount'] = np.exp(-discs_down['spot rate'] * discs_down.index)

    results_up = price_callable_bond(info, quotes, discs_up, KEY_CALLABLE)
    results_down = price_callable_bond(info, quotes, discs_down, KEY_CALLABLE)
    price_up = results_up['callable_price']
    price_down = results_down['callable_price']
    forward_price_up = results_up['forward_price']
    forward_price_down = results_down['forward_price']
    

    imp_vol_forward_price_up = calc_implied_vol_forward_price(info, quotes, discs_up, KEY_CALLABLE)
    imp_vol_forward_price_down = calc_implied_vol_forward_price(info, quotes, discs_down, KEY_CALLABLE)

    comparision_df = pd.DataFrame({
        'price': [price, price_up, price_down],
        'forward_price': [forward_price, forward_price_up, forward_price_down],
        'imp_vol_forward_price': [iv_fwd_price, imp_vol_forward_price_up, imp_vol_forward_price_down]
    }, index=['base', 'up', 'down'])
    display(comparision_df)
    
    duration = (price_down - price_up) / (2 * price * shock) # -1/p * dp/dy
    return duration, comparision_df
    # return None, None



In [89]:
hypothetical_bond_duration = calc_modified_duration(callable_price, cpn_rate, cpn_freq, (date_matures - date_quoted).days / 365, N, ytm_never_called)
print(f"Modified duration of the bond if never called: {hypothetical_bond_duration:.4f}")

duration_from_shocks, comparision_df = calc_duration_from_shocks(callable_price, info, quotes, discs, shock = 0.0001)
print(f"Duration of callable bond calculated from shocks: {duration_from_shocks:.4f}")



Modified duration of the bond if never called: 4.8206


,price,forward_price,imp_vol_forward_price
base,96.519229,100.723182,0.044568
up,96.338234,100.624541,0.044675
down,96.407405,100.662769,0.044461


Duration of callable bond calculated from shocks: 3.5833


### 1.7.

Calculate the OAS of the `callable` bond.

How does it compare to the quoted OAS?

Recall that the OAS is the parallel shift in the spot curve needed to align the modeled value to the market quote.


In [105]:
def calc_OAS(info, quotes, discs, key):
    # calculate the option adjusted spread of a callable bond given its price and other characteristics
    # use a numerical method to solve for the spread that equates the price of the callable bond to the price of a vanilla bond with the same characteristics but discounted at the risk-free rate plus the spread
    cpn_rate = info.loc['Cpn Rate', key]
    cpn_freq = info.loc['Cpn Freq', key]
    date_quoted = info.loc['Date Quoted', key]
    date_matures = info.loc['Date Matures', key]
    N = 100 # face value of bond
    callable_price = price_callable_bond(info, quotes, discs, key)['callable_price']

    def price_diff(spread):
        discs_copy = discs.copy()
        discs_copy['spot rate'] = discs_copy['spot rate'] + spread
        discs_copy['discount'] = np.exp(-discs_copy['spot rate'] * discs_copy.index)
        vanilla_price = price_vanilla_bond(cpn_rate, cpn_freq, (date_matures - date_quoted).days / 365, N, discs_copy)
        return vanilla_price - callable_price
    result = root_scalar(price_diff, bracket=[-0.5, 0.5]) # search for OAS between 1 basis point and 1000 basis points
    oas = result.root
    return oas

oas = calc_OAS(info, quotes, discs, KEY_CALLABLE)
print(f"Option Adjusted Spread (OAS) of the callable bond: {oas*1e4:.4f}bps")

Option Adjusted Spread (OAS) of the callable bond: 64.9064bps


### 1.8. Optional OTM Callables


There are a few other Freddie Mac callables that may be of interest.
* `FHLMC 0.97 01/28/28`
* `FHLMC 1.25 01/29/30`

Though these are technically callable, they are far out of the money (OTM). 
* Expiring in 3 months, though code below changes it to 6 monhts, to match coupon.
* These don't have interesting convexity due to being so far OTM.


In [106]:
KEY_CALLABLE1 = 'FHLMC 1 1/4 01/29/30'
KEY_CALLABLE2 = 'FHLMC 0.97 01/28/28'

OAS1 = calc_OAS(info, quotes, discs, KEY_CALLABLE1)
OAS2 = calc_OAS(info, quotes, discs, KEY_CALLABLE2)

print(f"OAS of {KEY_CALLABLE1}: {OAS1*1e4:.4f}bps")
print(f"OAS of {KEY_CALLABLE2}: {OAS2*1e4:.4f}bps")

Using nearest discount factor for t=0.2054794520547945
Using nearest discount factor for t=0.2054794520547945
Using nearest discount factor for t=0.20273972602739726
Using nearest discount factor for t=0.20273972602739726
OAS of FHLMC 1 1/4 01/29/30: -4.2140bps
OAS of FHLMC 0.97 01/28/28: -4.2492bps


In [107]:
info.style.format('{:.2%}',subset=pd.IndexSlice[["Cpn Rate"], :]).format('{:,.0f}',subset=pd.IndexSlice[["Amount Issued"], :]).format('{:%Y-%m-%d}',subset=pd.IndexSlice[["Date Quoted","Date Issued","Date Matures","Date Next Call","Date of First Possible Call"], :])

,FHLMC 0.97 01/28/28,FHLMC 1 1/4 01/29/30,FHLMC 4.41 01/28/30
info,,,
CUSIP,3134GW5F9,3134GWGK6,3134HA4V2
Issuer,FREDDIE MAC,FREDDIE MAC,FREDDIE MAC
Maturity Type,CALLABLE,CALLABLE,CALLABLE
Issuer Industry,GOVT AGENCY,GOVT AGENCY,GOVT AGENCY
Amount Issued,"30,000,000","25,000,000","10,000,000"
Cpn Rate,0.97%,1.25%,4.41%
Cpn Freq,2,2,2
Date Quoted,2025-02-13,2025-02-13,2025-02-13
Date Issued,2020-10-28,2020-07-29,2025-01-28


In [108]:
quotes.style.format('{:.2f}', subset=pd.IndexSlice[quotes.index[1:], :]).format('{:%Y-%m-%d}', subset=pd.IndexSlice['Date Quoted', :])

,FHLMC 0.97 01/28/28,FHLMC 1 1/4 01/29/30,FHLMC 4.41 01/28/30
quotes,,,
Date Quoted,2025-02-13,2025-02-13,2025-02-13
TTM,2.95,4.96,4.96
Clean Price,90.14,85.11,99.89
Dirty Price,90.19,85.16,100.09
Accrued Interest,0.04,0.05,0.20
YTM Call,54.24,85.40,4.45
YTM Maturity,4.57,4.65,4.43
Duration,2.92,4.81,4.50
Modified Duration,2.85,4.70,4.40


### 1.9. ATM with 1-yr expiry

Try this alternate file `2025-02-18` for a recently-issued bond of size $1bn with a one-year expiration.
* Easier to see the negative convexity.
* Large size, recency should be more liquid.


In [109]:
FILE_BOND = '../data/callable_bonds_2025-02-18.xlsx'
FILE_CURVE = '../data/discount_curve_2025-02-18.xlsx'
KEY_CALLABLE = 'FHLMC 4.55 02/11/28'

In [113]:
results.keys()

dict_keys(['callable_price', 'option_price', 'vanilla_bond_price', 'forward_price'])

In [115]:
info = pd.read_excel(FILE_BOND,sheet_name='info').set_index('info')
quotes = pd.read_excel(FILE_BOND,sheet_name='quotes').set_index('quotes')
# discs = pd.read_excel(FILE_CURVE,sheet_name='discount curve').set_index('ttm')
OAS = calc_OAS(info, quotes, discs, KEY_CALLABLE)
print(f"OAS of {KEY_CALLABLE}: {OAS*1e4:.4f}bps")

results = price_callable_bond(info, quotes, discs, KEY_CALLABLE)
callable_price = results['callable_price']
forward_price = results['forward_price']
vanilla_bond_price = results['vanilla_bond_price']
option_price = results['option_price']
implied_volatility_forward_price = calc_implied_vol_forward_price(info, quotes, discs, KEY_CALLABLE)


print(f"Callable Bond Price: {results['callable_price']:.2f}")
print(f"Forward Price: {results['forward_price']:.2f}")
print(f"Vanilla Bond Price: {results['vanilla_bond_price']:.2f}")
print(f"Option Price: {results['option_price']:.2f}")
print(f"Implied Volatility of Forward Price: {implied_volatility_forward_price:.4f}")

print(f"quoted clean price: {quotes.loc['Clean Price', KEY_CALLABLE]:.2f} vs calculated price: {callable_price:.2f}")

OAS of FHLMC 4.55 02/11/28: 55.9413bps
Callable Bond Price: 97.50
Forward Price: 101.06
Vanilla Bond Price: 99.17
Option Price: 1.67
Implied Volatility of Forward Price: 0.0284
quoted clean price: 99.72 vs calculated price: 97.50
